In [1]:
import undetected_chromedriver as uc
import pandas as pd
import time
import re
from selenium.webdriver.common.by import By

In [2]:
# 1. Expanded search queries to cover the entire data market
search_queries = [
    'Data Analyst', 
    'Data Scientist', 
    'Data Engineer', 
    'Business Analyst',
    'BI Developer',
    'Machine Learning Engineer',
    'Data Architect'
]

# 2. Filtering keywords to keep the dataset perfectly clean
invalid_keywords = ['hr', 'call center', 'sales', 'customer service', 'receptionist']
valid_keywords = ['data', 'analyst', 'scientist', 'engineer', 'business', 'bi', 'machine learning', 'sql', 'python']

def is_relevant_job(title):
    title_lower = title.lower()
    if any(invalid in title_lower for invalid in invalid_keywords): 
        return False
    if any(valid in title_lower for valid in valid_keywords): 
        return True
    return False

print(" Start Scraping ")

options = uc.ChromeOptions()
try:
    # Forcing version 145 to match your local Chrome version
    driver = uc.Chrome(options=options, version_main=145)
except Exception as e:
    print(f"[!] Failed to start driver: {e}")
    exit()

jobs_data = []
# Set a target PER QUERY ( 300 jobs )
target_per_query = 300  

print(f"[*] Scraping started. Target: {target_per_query} jobs per category.")

try:
    for query in search_queries:
        page = 1
        jobs_collected_for_this_query = 0
        
        # Loop until we hit the specific target for THIS current query
        while jobs_collected_for_this_query < target_per_query:
            url = f'https://egypt.tanqeeb.com/jobs/search?keywords={query.replace(" ", "+")}&page={page}'
            print(f"\n[*] Fetching: '{query}' - Page: {page}...")
            
            driver.get(url)
            # Wait for Cloudflare validation and page load
            time.sleep(4) 
            
            try:
                cards = driver.find_elements(By.CSS_SELECTOR, "div.card-list")
            except:
                cards = []
                
            if not cards:
                print(f"[!] No more jobs found for '{query}'. Moving to next category.")
                break
                
            # Step A: Extract basic info and job links
            job_links = []
            for card in cards:
                try:
                    title_elem = card.find_element(By.CSS_SELECTOR, "h2.hover-title")
                    title = title_elem.text.strip()
                    
                    if not is_relevant_job(title): 
                        continue
                    
                    try:
                        company = card.find_element(By.CSS_SELECTOR, "span.job-meta-company").text.strip()
                    except: 
                        company = "Not Specified"
                    
                    try:
                        location = card.find_element(By.CSS_SELECTOR, "span.job-meta-item").text.strip()
                    except: 
                        location = "Not Specified"
                    
                    try:
                        link_elem = card.find_element(By.CSS_SELECTOR, "a.card-list-item")
                        link = link_elem.get_attribute("href")
                        if link:
                            job_links.append({
                                'title': title, 'company': company, 'location': location, 'link': link
                            })
                    except: 
                        pass
                except Exception as e:
                    continue
            
            # Step B: Visit each specific job link to extract the real description
            for job in job_links:
                # Stop if we hit the target for this specific query
                if jobs_collected_for_this_query >= target_per_query: 
                    break
                
                driver.get(job['link'])
                time.sleep(3) # Wait for JavaScript to render the text
                
                full_description = "Not Specified"
                experience_level = "Not Specified"
                
                try:
                    # Extract ALL visible text from the page body
                    body_element = driver.find_element(By.TAG_NAME, "body")
                    page_text = body_element.text
                    
                    if page_text and len(page_text) > 50:
                        clean_text = " ".join(page_text.split())
                        full_description = clean_text
                        
                        # Extract Experience using Regex
                        exp_matches = re.findall(r'(?i)experience[^\w]{0,5}(.*?\d+.*?(?:years|yrs)|.*?level)', clean_text)
                        if exp_matches:
                            experience_level = exp_matches[0].strip()
                except Exception as e:
                    pass
                    
                jobs_data.append({
                    'Job Title': job['title'],
                    'Company': job['company'],
                    'Location': job['location'],
                    'Experience': experience_level,
                    'Full Description (Skills)': full_description[:3000] if full_description.strip() else "Not Specified",
                    'Job Link': job['link'],
                    'Search Query': query
                })
                
                jobs_collected_for_this_query += 1
                
            print(f"[+] '{query}' progress: {jobs_collected_for_this_query}/{target_per_query} jobs collected.")
            page += 1

finally:
    driver.quit()

# Convert to Pandas DataFrame and save
df = pd.DataFrame(jobs_data)
csv_filename = 'tanqeeb_jobs_dataset.csv'
df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

print(f"\n[+] Done! dataset successfully saved to '{csv_filename}'")
print(f"[+] Total jobs collected : {len(df)}")

 Start Scraping 
[*] Scraping started. Target: 300 jobs per category.

[*] Fetching: 'Data Analyst' - Page: 1...
[+] 'Data Analyst' progress: 15/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 2...
[+] 'Data Analyst' progress: 30/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 3...
[+] 'Data Analyst' progress: 45/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 4...
[+] 'Data Analyst' progress: 60/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 5...
[+] 'Data Analyst' progress: 75/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 6...
[+] 'Data Analyst' progress: 90/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 7...
[+] 'Data Analyst' progress: 105/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 8...
[+] 'Data Analyst' progress: 120/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 9...
[+] 'Data Analyst' progress: 135/300 jobs collected.

[*] Fetching: 'Data Analyst' - Page: 10...
[+] 'Data Analyst' progress: